# Task 2 — Data Cleaning and Basic Analysis

**SWYNEX Bioinformatics Internship**  
**Dataset:** GSE19804 — lung cancer gene expression  
**Source:** [NCBI GEO GSE19804](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE19804)

In this notebook, I check the dataset for missing or invalid values and duplicate probe or sample IDs. I also check the sample information, including the matched tumor and non-tumor samples, and calculate basic statistics.

The data are already processed (log2 RMA values), so I do not normalize them again.

## 1. Load the data

First, I load the GSE19804 dataset from NCBI GEO. If the file is not already available, it is downloaded automatically.

The dataset contains probe sets in rows and samples in columns.

In [6]:
import os
import gzip
import urllib.request
import pandas as pd
import numpy as np

file_name = "GSE19804_series_matrix.txt.gz"
url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE19nnn/GSE19804/matrix/GSE19804_series_matrix.txt.gz"

if not os.path.exists(file_name):
    urllib.request.urlretrieve(url, file_name)

expression = pd.read_csv(file_name, sep="\t", compression="gzip",
                         comment="!", index_col=0)
print("Rows (probe sets):", expression.shape[0])
print("Columns (samples):", expression.shape[1])
expression.iloc[:5, :5]

Rows (probe sets): 54675
Columns (samples): 120


,GSM494556,GSM494557,GSM494558,GSM494559,GSM494560
ID_REF,,,,,
1007_s_at,11.96280,11.38780,11.19280,11.90360,11.75230
1053_at,7.05652,7.33711,7.26121,7.91211,8.27448
117_at,6.55585,6.96023,7.14488,7.13887,7.72479
121_at,8.43850,8.51636,8.43138,8.67404,8.78145
1255_g_at,3.73169,3.77978,3.99853,3.69171,3.93509


## 2. Check sample information

The GEO file contains information about the samples. I use it to count the tumor and non-tumor samples, check the patient pairs, and compare the sample IDs with the expression data.

In [7]:
# Read sample information from the GEO file
with gzip.open(file_name, "rt") as file:
    for line in file:
        if line.startswith("!Sample_geo_accession\t"):
            sample_ids = [x.strip('"') for x in line.strip().split("\t")[1:]]
        if line.startswith("!Sample_title\t"):
            titles = [x.strip('"') for x in line.strip().split("\t")[1:]]
        if line.startswith("!series_matrix_table_begin"):
            break

samples = pd.DataFrame({"sample_id": sample_ids, "title": titles})

samples["tissue"] = samples["title"].str.split().str[1].replace(
    {"Cancer": "Tumor", "Normal": "Non-tumor"}
)
samples["patient_id"] = samples["title"].str.split().str[2].str[:-1]

# Count tumor and non-tumor samples
print(samples["tissue"].value_counts())

# Check patient pairs
pairs = samples.groupby("patient_id")["tissue"].nunique()
counts = samples.groupby("patient_id").size()

complete_pairs = (pairs == 2) & (counts == 2)

print("Complete patient pairs:", complete_pairs.sum())
print("All patients have complete pairs:", complete_pairs.all())

# Compare sample IDs with the expression data
print("Sample IDs match:",
      samples["sample_id"].tolist() == expression.columns.tolist())

samples.head()


tissue
Tumor        60
Non-tumor    60
Name: count, dtype: int64
Complete patient pairs: 60
All patients have complete pairs: True
Sample IDs match: True


,sample_id,title,tissue,patient_id
0,GSM494556,Lung Cancer 2T,Tumor,2
1,GSM494557,Lung Cancer 3T,Tumor,3
2,GSM494558,Lung Cancer 6T,Tumor,6
3,GSM494559,Lung Cancer 17T,Tumor,17
4,GSM494560,Lung Cancer 32T,Tumor,32


## 3. Check data quality

I check the expression data for missing values, non-numeric values, infinite values, and duplicate IDs.

These checks help me decide whether any data cleaning is needed before the analysis.

In [8]:
# Convert expression values to numbers
cleaned = expression.apply(pd.to_numeric, errors="coerce")

# Check missing and non-numeric values
missing = expression.isna().sum().sum()
non_numeric = cleaned.isna().sum().sum() - missing

# Check infinite values
infinite = np.isinf(cleaned.to_numpy()).sum()

# Check duplicate IDs
duplicate_probes = expression.index.duplicated().sum()
duplicate_samples = samples["sample_id"].duplicated().sum()

print("Missing values:", missing)
print("Non-numeric values:", non_numeric)
print("Infinite values:", infinite)
print("Duplicate probe IDs:", duplicate_probes)
print("Duplicate sample IDs:", duplicate_samples)

# Check if cleaning is needed
if missing + non_numeric + infinite + duplicate_probes + duplicate_samples == 0:
    print("No data cleaning was needed.")
else:
    print("Data quality problems were found. Review the data before continuing.")


Missing values: 0
Non-numeric values: 0
Infinite values: 0
Duplicate probe IDs: 0
Duplicate sample IDs: 0
No data cleaning was needed.


## 4. Basic statistics

I calculate basic statistics for the gene expression data, including the mean, median, standard deviation, minimum, and maximum.

I also summarize the values for each sample and compare the average sample statistics between the tumor and non-tumor groups.

In [9]:
# Overall statistics
all_values = pd.Series(cleaned.to_numpy().ravel())

print("Total expression values:", len(all_values))
print("\nOverall expression summary:")
print(all_values.describe().drop("count").round(4))

# Statistics for each sample
sample_stats = cleaned.agg(["mean", "median", "std", "min", "max"]).T
sample_stats.index.name = "sample_id"

sample_stats = sample_stats.join(
    samples.set_index("sample_id")[["patient_id", "tissue"]]
)

print("\nFirst five samples:")
print(sample_stats.head().round(4))

# Average statistics for each tissue group
group_stats = sample_stats.groupby("tissue")[
    ["mean", "median", "std"]
].mean().round(4)

print("\nAverage sample statistics by tissue:")
print(group_stats)


Total expression values: 6561000

Overall expression summary:
mean     6.4259
std      2.1424
min      3.0427
25%      4.6878
50%      6.0284
75%      7.8535
max     14.8903
dtype: float64

First five samples:
             mean  median     std     min      max patient_id tissue
sample_id                                                           
GSM494556  6.4259  6.0284  2.1424  3.0427  14.8903          2  Tumor
GSM494557  6.4259  6.0284  2.1424  3.0427  14.8903          3  Tumor
GSM494558  6.4259  6.0278  2.1424  3.0427  14.8903          6  Tumor
GSM494559  6.4259  6.0292  2.1424  3.0427  14.8903         17  Tumor
GSM494560  6.4259  6.0284  2.1424  3.0427  14.8903         32  Tumor

Average sample statistics by tissue:
             mean  median     std
tissue                           
Non-tumor  6.4259  6.0284  2.1424
Tumor      6.4259  6.0284  2.1424


## 5. Save the results

I save the sample information and basic statistics as CSV files so they can be used later. The original dataset remains unchanged.

In [10]:

# Save the results as CSV files
samples.to_csv("sample_information.csv", index=False)
sample_stats.to_csv("sample_statistics.csv")
group_stats.to_csv("group_statistics.csv")

print("Results saved successfully.")

Results saved successfully.


## 6. Conclusion

The GSE19804 dataset contains 54,675 probe sets and 120 samples from 60 patients, with matched tumor and non-tumor samples.

The data quality checks found no missing, non-numeric, or infinite values, and no duplicate probe or sample IDs. Therefore, no values needed to be removed, and no additional normalization was performed.

The overall mean expression was 6.4259, and the median was 6.0284. Basic statistics were also calculated for individual samples and the tumor and non-tumor groups. The sample information and statistical summaries were saved as CSV files.

These results provide a basic summary of the processed gene expression data.